# Tomo — CG vs FBP Self-Test

End-to-end verification of `Tomo` on a 3-D Shepp-Logan phantom:
1. Build phantom, forward-project to sinogram via `R`.
2. Reconstruct with **FBP** (`fbp`, ramp/shepp filters).
3. Reconstruct with **CG** (`rec_tomo`).
4. Compare residuals and display slices.

In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from holotomocupy.tomo import Tomo
from holotomocupy.utils import mshow, mshow_complex

## Parameters

In [ ]:
n      = 256          # object / detector size (pixels)
nz     = 16           # number of z-slices
ntheta = 1024          # number of projection angles
mask_r = 1          # circular support mask radius

theta = np.linspace(0, np.pi, ntheta, endpoint=False, dtype='float32')

## Phantom — Shepp-Logan

Classic 2-D Shepp-Logan ellipsoid model replicated across `nz` slices.

In [ ]:
def shepp_logan_phantom(n):
    """Return an (n, n) float32 Shepp-Logan phantom."""
    # 10 ellipses: [A, a, b, x0, y0, phi_deg]
    ellipses = [
        [ 0.2,  0.69,  0.92,   0,      0,     0  ],
        [-0.18, 0.6624,0.874,  0,    -0.0184,  0  ],
        [-0.02, 0.11,  0.31,   0.22,   0,    -18  ],
        [-0.02, 0.16,  0.41,  -0.22,   0,     18  ],
        [ 0.01, 0.21,  0.25,   0,      0.35,   0  ],
        [ 0.01, 0.046, 0.046,  0,      0.1,    0  ],
        [ 0.02, 0.046, 0.046,  0,     -0.1,    0  ],
        [ 0.01, 0.046, 0.023, -0.08,  -0.605,  0  ],
        [ 0.01, 0.023, 0.023,  0,     -0.606,  0  ],
        [ 0.01, 0.023, 0.046,  0.06,  -0.605,  0  ],
    ]
    t  = np.linspace(-1, 1, n, endpoint=False, dtype='float64')
    x, y = np.meshgrid(t, t, indexing='ij')
    img  = np.zeros((n, n), dtype='float32')
    for A, a, b, x0, y0, phi in ellipses:
        c, s  = np.cos(np.deg2rad(phi)), np.sin(np.deg2rad(phi))
        xr    = c * (x - x0) + s * (y - y0)
        yr    = -s * (x - x0) + c * (y - y0)
        img  += A * ((xr / a)**2 + (yr / b)**2 <= 1).astype('float32')
    return img

sl2d   = shepp_logan_phantom(n)
obj_np = np.stack([sl2d] * nz, axis=0)         # [nz, n, n]
obj    = cp.array(obj_np)

mshow(obj[nz // 2], True)

## Initialise Tomo and Forward-Project

In [ ]:
cl = Tomo(n, nz, theta, mask_r)

sino = cl.R(obj)   # [ntheta, nz, n]
print(f'sinogram shape: {sino.shape},  dtype: {sino.dtype}')
mshow(sino[:, nz // 2], True)

## FBP Reconstruction

In [ ]:
rec_fbp_ramp  = cl.fbp(sino, filter_name='ramp')
rec_fbp_shepp = cl.fbp(sino, filter_name='shepp')

print(f'FBP (ramp)  range: {rec_fbp_ramp[nz//2].min():.4f} … {rec_fbp_ramp[nz//2].max():.4f}')
print(f'FBP (shepp) range: {rec_fbp_shepp[nz//2].min():.4f} … {rec_fbp_shepp[nz//2].max():.4f}')

mshow(rec_fbp_ramp[nz // 2],  True)
mshow(rec_fbp_shepp[nz // 2], True)

## CG Reconstruction

In [ ]:
niter_cg = 64
rec_cg   = cl.rec_tomo(sino, niter=niter_cg)

print(f'CG ({niter_cg} iter) range: {rec_cg[nz//2].min():.4f} … {rec_cg[nz//2].max():.4f}')
mshow(rec_cg[nz // 2], True)

## Comparison

In [ ]:
iz = nz // 2   # slice to display
gt = obj[iz].get()

fbp_r = rec_fbp_ramp[iz].get()
fbp_s = rec_fbp_shepp[iz].get()
cg    = rec_cg[iz].get()

# Scale reconstructions to match GT amplitude (FBP is off by a global constant)
scale_fbp_r = gt.max() / fbp_r.max()
scale_fbp_s = gt.max() / fbp_s.max()
scale_cg    = gt.max() / cg.max()

fbp_r_sc = fbp_r * scale_fbp_r
fbp_s_sc = fbp_s * scale_fbp_s
cg_sc    = cg    * scale_cg

err_fbp_r = np.linalg.norm(gt - fbp_r_sc) / np.linalg.norm(gt)
err_fbp_s = np.linalg.norm(gt - fbp_s_sc) / np.linalg.norm(gt)
err_cg    = np.linalg.norm(gt - cg_sc)    / np.linalg.norm(gt)

print(f'Relative L2 error:')
print(f'  FBP (ramp)  : {err_fbp_r:.4f}')
print(f'  FBP (shepp) : {err_fbp_s:.4f}')
print(f'  CG ({niter_cg} iter): {err_cg:.4f}')

vmin, vmax = gt.min(), gt.max()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Ground truth', f'FBP ramp (err={err_fbp_r:.3f})', f'FBP shepp (err={err_fbp_s:.3f})', f'CG {niter_cg}it (err={err_cg:.3f})']
images = [gt, fbp_r_sc, fbp_s_sc, cg_sc]
for ax, img, title in zip(axes, images, titles):
    im = ax.imshow(img, cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Ground truth', f'FBP ramp (err={err_fbp_r:.3f})', f'FBP shepp (err={err_fbp_s:.3f})', f'CG {niter_cg}it (err={err_cg:.3f})']
images = [gt, fbp_r_sc, fbp_s_sc, cg_sc]
for ax, img, title in zip(axes, images, titles):
    im = ax.imshow(img-gt, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## Adjoint Test: `<R(x), y> == <x, RT(y)>`

In [ ]:
rng = np.random.default_rng(42)
x   = cp.array(rng.standard_normal((nz, n, n)).astype('float32'))
y   = cp.array(rng.standard_normal((ntheta, nz, n)).astype('float32'))

Rx  = cl.R(x)
RTy = cl.RT(y)

lhs = float(cp.real(cp.dot(Rx.ravel(), y.ravel())))
rhs = float(cp.real(cp.dot(x.ravel(), RTy.ravel())))

print(f'<R(x), y>   = {lhs:.6e}')
print(f'<x, RT(y)>  = {rhs:.6e}')
print(f'relative difference = {abs(lhs - rhs) / abs(lhs):.2e}')